## ECommerce Sales Analysis

### Import Libraries

In [1]:
import pandas as pd

### Dataset Loaded

In [2]:
# Load dataset
df = pd.read_csv('C:/Users/komal/Desktop/Power BI/ECommerce Sales/Online-eCommerce.csv')
print('Dataset loaded successfully')

# 1. Shape of dataset
print('Shape:', df.shape)
print()

# 2. Column names
print('Columns:', df.columns.tolist())
print()

# 3. First 5 rows
print('First 5 rows:', df.head())
print()

# 4. Data Types
print('Data Types:', df.dtypes)
print()

# 5. Null values
print('Null values:', df.isnull().sum())
print()

# 6. Basic
print('Basic Statistics:', df[['Cost','Sales', 'Quantity','Total_Cost','Total_Sales']].describe())
print()

# 7. Check categorical columns
for col in ['Status', 'Category','Brand', 'State_Code']:
    print(f'Unique [{col}]: {df[col].unique()}')
    print()

# 8. Date range
print('Date range:', df['Order_Date'].min(), 'to', df['Order_Date'].max())

Dataset loaded successfully
Shape: (5110, 14)

Columns: ['Order_Number', 'State_Code', 'Customer_Name', 'Order_Date', 'Status', 'Product', 'Category', 'Brand', 'Cost', 'Sales', 'Quantity', 'Total_Cost', 'Total_Sales', 'Assigned Supervisor']

First 5 rows:    Order_Number State_Code   Customer_Name  Order_Date     Status  \
0      139374.0         AP     Adhir Samal  2020-01-11  Delivered   
1      139375.0         AP  Dannana Jhammi  2020-01-11  Delivered   
2      139376.0         AS     Vipin Kumar  2020-01-11  Delivered   
3      139377.0         BR   Ranjeet Kumar  2020-01-11  Delivered   
4      139378.0         CG   Sajal Singhal  2020-01-11      Order   

                    Product      Category     Brand    Cost    Sales  \
0                512 GB M.2           SSD   Samsung  6500.0   8450.0   
1      RYZEN 3rd gen. 3500            CPU     Intel  8500.0  11050.0   
2          2GB Graphic Card  Graphic Card    Nvidia  7000.0   9100.0   
3            16 GB DDR4 RAM           RAM

TypeError: '<=' not supported between instances of 'str' and 'float'

### Cleaning

In [46]:
# Drop Blank Rows
df = df.dropna(how='all')

# Fix category data
df['Category'] = df['Category'].replace('Motherboard','MotherBoard')
df['Category'] = df['Category'].replace('Computer Case', 'Cabinet')

# Convert Order Date to datetime
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

# Fix data types
df['Order_Number'] = df['Order_Number'].astype(int)
df['Quantity'] = df['Quantity'].astype(int)
df['Cost'] = df['Cost'].astype(int)
df['Sales'] = df['Sales'].astype(int)
df['Total_Cost'] = df['Total_Cost'].astype(int)
df['Total_Sales'] = df['Total_Sales'].astype(int)

# Rename
df = df.rename(columns={'Assigned Supervisor': 'Supervisor'})

# 
print('Shape:',df.shape)
print('Dtypes:', df.dtypes)
print('Nulls:', df.isnull().sum())


# Save as CSV for MySQL 
df.to_csv('ecommerce_clean.csv', index=False)
print('File Saved: ecommerce_clean.csv')

Shape: (5095, 14)
Dtypes: Order_Number              int64
State_Code               object
Customer_Name            object
Order_Date       datetime64[ns]
Status                   object
Product                  object
Category                 object
Brand                    object
Cost                      int64
Sales                     int64
Quantity                  int64
Total_Cost                int64
Total_Sales               int64
Supervisor               object
dtype: object
Nulls: Order_Number     0
State_Code       0
Customer_Name    0
Order_Date       0
Status           0
Product          0
Category         0
Brand            0
Cost             0
Sales            0
Quantity         0
Total_Cost       0
Total_Sales      0
Supervisor       0
dtype: int64
File Saved: ecommerce_clean.csv


In [30]:
# CHECK 1: Strip whitespace from string columns
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].str.strip()
print("Whitespace stripped")

# CHECK 2: Convert Order_Date to datetime
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
print("Order_Date dtype:", df['Order_Date'].dtype)

df['Product'] = df['Product'].str.replace(r'[\\"]', '', regex=True)
df['Product'] = df['Product'].str.strip()

# CHECK 3: Final confirmation
print("Final Shape:", df.shape)
print("Nulls:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
print("\nDone. Data is clean.")

Whitespace stripped
Order_Date dtype: datetime64[ns]
Final Shape: (5095, 19)
Nulls: 0
Duplicates: 0

Done. Data is clean.


### Feature Engineering

In [6]:
df = pd.read_csv('C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/cleaned_ecommerce.csv')

df['Order_Date'] = pd.to_datetime(df['Order_Date'])

str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].str.strip()


# 1. Profit
df['Profit'] = df['Total_Sales'] - df['Total_Cost']

# 2. Profit Margin
df['Profit_Margin_Pct'] = round((df['Profit'] / df['Total_Sales']) * 100, 2)

# 3. Order Year
df['Order_Year'] = df['Order_Date'].dt.year

# 4. Order Month
df['Order_Month'] = df['Order_Date'].dt.month

# 5. Checking Deliver
df['Is_Delivered'] = (df['Status'] == 'Delivered').astype(int)

print("Final Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nSample:")
print(df[['Profit', 'Profit_Margin_Pct', 'Order_Year', 'Order_Month', 'Is_Delivered']].head(5))

# Verifying
print(df['Profit'].describe())
print()

print(df['Profit_Margin_Pct'].describe())
print()

print(df['Is_Delivered'].describe())
print()

print(df['Order_Year'].value_counts().sort_index())
print()

print('Shape:', df.shape)

# Diagnostic
print("Sample of key columns:")
print(df[['Total_Cost', 'Total_Sales', 'Profit', 'Profit_Margin_Pct']].head(20))

print("\nTotal_Sales dtype:", df['Total_Sales'].dtype)
print("Total_Cost dtype:", df['Total_Cost'].dtype)
print("Profit dtype:", df['Profit'].dtype)

print("\nUnique Profit_Margin_Pct values:")
print(df['Profit_Margin_Pct'].unique())

Final Shape: (5095, 19)
Columns: ['Order_Number', 'State_Code', 'Customer_Name', 'Order_Date', 'Status', 'Product', 'Category', 'Brand', 'Cost', 'Sales', 'Quantity', 'Total_Cost', 'Total_Sales', 'Supervisor', 'Profit', 'Profit_Margin_Pct', 'Order_Year', 'Order_Month', 'Is_Delivered']

Sample:
   Profit  Profit_Margin_Pct  Order_Year  Order_Month  Is_Delivered
0    1950              23.08        2020            1             1
1    7650              23.08        2020            1             1
2    4200              23.08        2020            1             1
3    5895              23.08        2020            1             1
4    9180              23.08        2020            1             0
count     5095.000000
mean      4497.533464
std       3772.475528
min        105.000000
25%       1350.000000
50%       3216.000000
75%       6885.000000
max      17400.000000
Name: Profit, dtype: float64

count    5.095000e+03
mean     2.308000e+01
std      3.553062e-15
min      2.308000e+01
25% 

In [9]:
df['Profit'] = df['Total_Sales'] - df['Total_Cost']
df['Profit_Margin_Pct'] = round((df['Profit'] /df['Total_Sales']) * 100, 2)
df['Order_Year'] = df['Order_Date'].dt.year
df['Order_Month'] = df['Order_Date'].dt.month
df['Is_Delivered'] = (df['Status'] == 'Delivered').astype(int)

print("Data Ready")

Data Ready


### Business Question

In [12]:
# Q1 — What is Total Revenue, Cost and Profit?

total_revenue = df['Total_Sales'].sum()
total_cost = df['Total_Cost'].sum()
total_profit = df['Profit'].sum()

print('Total Revenue:',total_revenue)
print('Total_Cost:', total_cost)
print('Total Profit:', total_profit)

print('Profit Margin:', round((total_profit / total_revenue) * 100, 2), "%")

Total Revenue: 99298043
Total_Cost: 76383110
Total Profit: 22914933
Profit Margin: 23.08 %


In [14]:
# Q2. Which Category Generates the Most profit?

category_profit = df.groupby('Category')['Profit'].sum().reset_index()
category_profit = category_profit.sort_values('Profit', ascending=False)

print(category_profit)

        Category   Profit
5        Monitor  5376255
0            CPU  4329300
2   Graphic Card  3026100
3            HDD  2973750
11           SSD  2351850
1        Cabinet  1122828
7          Mouse   884283
6    MotherBoard   729999
10           RAM   728007
9        Printer   663012
8            NIC   408804
4       Keyboard   320745


In [16]:
# Q3. Which state genertaes the most revenue?

state_revenue =df.groupby('State_Code').agg(
    Total_Orders = ('Order_Number', 'count'),
    Total_Revenue = ('Total_Sales', 'sum'),
    Total_Profit = ('Profit', 'sum')
).reset_index()

state_revenue = state_revenue.sort_values('Total_Revenue', ascending=False).head(10)
print(state_revenue)

   State_Code  Total_Orders  Total_Revenue  Total_Profit
19         MH           904       17621084       4066404
33         UP           458        9264645       2137995
11         GJ           476        9137726       2108706
9          DL           249        5061953       1168143
4          BR           254        4862221       1122051
31         TR           187        3660657        844767
30         TN           185        3428763        791253
6          CH            95        1958697        452007
22         MP            86        1885910        435210
23         MZ            87        1874418        432558


In [20]:
# Q4. What is the overall fulfillment rate?
status_counts = df['Status'].value_counts().reset_index()
status_counts.columns = ['Status', 'Total_Orders']
status_counts['Percentage'] = round(status_counts['Total_Orders'] / len (df) *100, 2)
print(status_counts)

fulfillment_rate = round(df['Is_Delivered'].sum() / len(df) * 100, 2)

print('Overall Fulfillment Rate:', fulfillment_rate, "%")

       Status  Total_Orders  Percentage
0  Processing          1278       25.08
1       Order          1276       25.04
2     Shipped          1273       24.99
3   Delivered          1268       24.89
Overall Fulfillment Rate: 24.89 %


In [23]:
# Q5. Which brand has the highest profit?
brand_profit = df.groupby('Brand').agg(
    Total_Revenue = ('Total_Sales', 'sum'),
    Total_Profit = ('Profit', 'sum')
).reset_index()
brand_profit = brand_profit.sort_values('Total_Profit', ascending=False)

print(brand_profit)

              Brand  Total_Revenue  Total_Profit
5             Intel       18760300       4329300
8           Samsung       16166345       3730695
2              Dell       14235195       3285045
7            Nvidia       13113100       3026100
10  Western Digital        8050250       1857750
0              Acer        6558630       1513530
3          Gigabyte        5886855       1358505
4             Hynix        5538520       1278120
9           Seagate        4836000       1116000
6               MSI        3205254        739674
1              Asus        2947594        680214


In [27]:
# Q6. Which supervisor is driving the most sales?
supervisor = df.groupby('Supervisor').agg(
    Total_Orders = ('Order_Number', 'count'),
    Total_Revenue = ('Total_Sales', 'sum'),
    Total_Profit = ('Profit','sum'),
    Delivered_Orders = ('Is_Delivered', 'sum')
).reset_index()

supervisor['Fulfillment_Rate'] = round(supervisor['Delivered_Orders'] / supervisor['Total_Orders'] * 100, 2)
supervisor = supervisor.sort_values('Total_Revenue',ascending=False)
print(supervisor)

     Supervisor  Total_Orders  Total_Revenue  Total_Profit  Delivered_Orders  \
1   Aarvi Gupta           967       18685368       4312008               247   
3   Ajay Sharma           948       17801186       4107966               243   
5   Vijay Singh           794       15939950       3678450               193   
4  Roshan Kumar           798       15887079       3666249               190   
0    Aadil Khan           797       15730767       3630177               205   
2  Advika Joshi           791       15253693       3520083               190   

   Fulfillment_Rate  
1             25.54  
3             25.63  
5             24.31  
4             23.81  
0             25.72  
2             24.02  


In [28]:
# Q7. What is the Year-wise Revenue Trend?
yearly = df.groupby('Order_Year').agg(
    Total_Revenue = ('Total_Sales', 'sum'),
    Total_Cost = ('Total_Cost', 'sum'),
    Total_Profit = ('Profit', 'sum'),
    Total_Orders = ('Order_Number', 'count')
).reset_index()

print(yearly)

   Order_Year  Total_Revenue  Total_Cost  Total_Profit  Total_Orders
0        2020       32319599    24861230       7458369          1646
1        2021       33657455    25890350       7767105          1717
2        2022       33320989    25631530       7689459          1732


### Export Clean Data

In [45]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].str.strip()

df['Product'] = df['Product'].str.replace(r'[\\"]', '', regex=True)
df['Product'] = df['Product'].str.strip()


df['Profit']            = df['Total_Sales'] - df['Total_Cost']
df['Profit_Margin_Pct'] = round((df['Profit'] / df['Total_Sales']) * 100, 2)
df['Order_Year']        = df['Order_Date'].dt.year
df['Order_Month']       = df['Order_Date'].dt.month
df['Is_Delivered']      = (df['Status'] == 'Delivered').astype(int)


#df = df.drop(columns=['Cost', 'Sales', 'Customer_Name'])


# Final check
print(df.shape)
print('Nulls:',df.isnull().sum().sum())
print('Duplicates:',df.duplicated().sum())

# Export
df.to_csv('cleaned_ecommerce_data.csv', index=False)
print('File Exported')

(5095, 16)
Nulls: 0
Duplicates: 0
File Exported
